# Part 1 Preprocessing Workflow

Central controller and presentation dashboard for Part 1 preprocessing. Main logic stays in `.py` modules; this notebook runs stages, shows the data flow, and summarizes the final tensors before Part 2 training.

What this notebook covers:
- raw split/window
- denoise `phase_hampel`
- window denoised data
- train-only temporal-shift augmentation
- train-only normalization stats sidecars
- Dataset/DataLoader smoke checks for GRU input
- class distribution and class-weight inspection

What this notebook does not do:
- no model definition
- no training loop
- no optimizer/checkpoint
- no materialized Gaussian-noise dataset


In [13]:
from pathlib import Path
import importlib.util
import subprocess
import sys

import numpy as np
import pandas as pd

####### MAIN CONFIG - EDIT HERE
def find_part1_dir(start: Path) -> Path:
    for base in [start, *start.parents]:
        if (base / 'Denoise').is_dir() and (base / 'split_and_window.py').is_file():
            return base
        for child in base.iterdir():
            if child.is_dir() and (child / 'Denoise').is_dir() and (child / 'split_and_window.py').is_file():
                return child
    raise FileNotFoundError('Cannot locate Part 1 preprocessing directory')

PART1_DIR = find_part1_dir(Path.cwd())

PROJECT_ROOT = PART1_DIR.parent
RAW_SESSION_DIR = PROJECT_ROOT / 'Dataset_CSI_3D_v2' / 'session1'
PROCESSED_DIR = PART1_DIR / 'processed'
SPLIT_PATH = PROCESSED_DIR / 'sample_split' / 'split.csv'
SPLIT_WINDOW_SCRIPT = PART1_DIR / 'split_and_window.py'
DENOISE_SCRIPT = PART1_DIR / 'Denoise' / 'CSI_denoise.py'
AUGMENT_SCRIPT = PART1_DIR / 'Augmentation' / 'CSI_augment.py'
NORMALIZE_SCRIPT = PART1_DIR / 'Normalization' / 'CSI_normalize.py'
DATASET_SCRIPT = PART1_DIR / 'Dataset' / 'CSI_dataset.py'

##### REBUILD CONTROLS - EDIT CAREFULLY
OVERWRITE_DENOISE = True   # True when rebuilding processed/denoised/phase_hampel from raw; False to protect existing denoise output.
OVERWRITE_AUGMENT = True   # True when rebuilding raw_shift/phase_hampel_shift; required after changing stride/crop.

##### PROFILE CONTROLS - EDIT WHEN CHANGING DATA VARIANTS
DENOISE_PROFILES = ['phase_hampel']
WINDOW_PROFILES = ['raw', 'raw_shift', 'phase_hampel', 'phase_hampel_shift']

##### AUGMENTATION CONTROLS - CURRENT BALANCED SETUP
AUGMENT_STRIDE = 16
AUGMENT_MAX_CROPS = 4

##### NORMALIZATION + DATALOADER CONTROLS
NORMALIZE_EPS = 1e-6
DATASET_PROFILE = 'raw_shift'
BATCH_SIZE = 8
NUM_WORKERS = 0  # Keep 0 inside Jupyter/Windows; try 2 later in Part 2 training scripts if needed.
TARGET = 'pose'  # pose | cell | presence | center
NOISE_SIGMA = 0.01  # train-time only, not saved to data.npz

##### PART 2 SCENARIOS - PRESENTATION ONLY, NO TRAINING HERE
TRAINING_SCENARIOS = [
    ('raw', 0.0),
    ('raw_shift', 0.0),
    ('raw_shift', 0.01),
    ('phase_hampel', 0.0),
    ('phase_hampel_shift', 0.0),
    ('phase_hampel_shift', 0.01),
]

print('PART1_DIR:', PART1_DIR)
print('RAW_SESSION_DIR:', RAW_SESSION_DIR)
print('SPLIT_PATH:', SPLIT_PATH)


PART1_DIR: c:\Users\nguyen\Desktop\DL_V2\Phần 1 tiền xử lí
RAW_SESSION_DIR: c:\Users\nguyen\Desktop\DL_V2\Dataset_CSI_3D_v2\session1
SPLIT_PATH: c:\Users\nguyen\Desktop\DL_V2\Phần 1 tiền xử lí\processed\sample_split\split.csv


In [14]:
def load_module(name: str, path: Path):
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise ImportError(f'Cannot load {name}: {path}')
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

denoise = load_module('csi_denoise', DENOISE_SCRIPT)
augment = load_module('csi_augment', AUGMENT_SCRIPT)
normalize = load_module('csi_normalize', NORMALIZE_SCRIPT)
csi_dataset = load_module('csi_dataset', DATASET_SCRIPT)
print('Loaded:', denoise.__file__)
print('Loaded:', augment.__file__)
print('Loaded:', normalize.__file__)
print('Loaded:', csi_dataset.__file__)


Loaded: c:\Users\nguyen\Desktop\DL_V2\Phần 1 tiền xử lí\Denoise\CSI_denoise.py
Loaded: c:\Users\nguyen\Desktop\DL_V2\Phần 1 tiền xử lí\Augmentation\CSI_augment.py
Loaded: c:\Users\nguyen\Desktop\DL_V2\Phần 1 tiền xử lí\Normalization\CSI_normalize.py
Loaded: c:\Users\nguyen\Desktop\DL_V2\Phần 1 tiền xử lí\Dataset\CSI_dataset.py


## 0. Pipeline Dashboard

This section summarizes the Part 1 data flow before any stage is run. Use it to understand which artifacts are stored on disk and which transformations happen only at runtime.

Stored artifacts:
- `data.npz`: canonical window tensors, never overwritten by normalization/noise
- `normalization_stats.npz`: train-only z-score sidecar per profile
- `windows.csv`: window metadata for traceability
- `labels.json`: label-id mapping

Runtime-only behavior:
- normalization is applied inside `Dataset`
- Gaussian noise is applied only for `split='train'`
- GRU layout is produced as `(192, 192)` per sample


In [15]:
##### PIPELINE MAP AND SCENARIOS
pipeline_rows = [
    {'stage': 'Raw session', 'input': 'Dataset_CSI_3D_v2/session1', 'output': 'raw CSV files', 'stored': 'raw data', 'note': 'Do not edit raw data'},
    {'stage': 'Split + window raw', 'input': 'raw CSV files', 'output': 'processed/windows/raw/data.npz', 'stored': 'yes', 'note': 'one centered 192-frame window/sample'},
    {'stage': 'Denoise', 'input': 'raw CSV files', 'output': 'processed/denoised/phase_hampel', 'stored': 'yes', 'note': 'phase sanitization + Hampel profile'},
    {'stage': 'Window denoised', 'input': 'processed/denoised/phase_hampel', 'output': 'processed/windows/phase_hampel/data.npz', 'stored': 'yes', 'note': 'uses same split.csv'},
    {'stage': 'Temporal shift', 'input': 'raw + phase_hampel windows', 'output': 'raw_shift + phase_hampel_shift', 'stored': 'yes', 'note': 'train only; val/test copied unchanged'},
    {'stage': 'Normalization stats', 'input': 'X_train per profile', 'output': 'normalization_stats.npz', 'stored': 'sidecar', 'note': 'fit train only; no normalized data.npz'},
    {'stage': 'Dataset/DataLoader', 'input': 'data.npz + normalization_stats.npz', 'output': '(B, 192, 192)', 'stored': 'runtime only', 'note': 'normalization + optional train noise'},
]
pipeline_df = pd.DataFrame(pipeline_rows)
display(pipeline_df)

scenario_df = pd.DataFrame([
    {
        'scenario': index + 1,
        'profile': profile,
        'denoise': 'phase_hampel' in profile,
        'temporal_shift': profile.endswith('_shift'),
        'noise_sigma': noise,
        'stored_data': f'processed/windows/{profile}/data.npz',
    }
    for index, (profile, noise) in enumerate(TRAINING_SCENARIOS)
])
display(scenario_df)


,stage,input,output,stored,note
0,Raw session,Dataset_CSI_3D_v2/session1,raw CSV files,raw data,Do not edit raw data
1,Split + window raw,raw CSV files,processed/windows/raw/data.npz,yes,one centered 192-frame window/sample
2,Denoise,raw CSV files,processed/denoised/phase_hampel,yes,phase sanitization + Hampel profile
3,Window denoised,processed/denoised/phase_hampel,processed/windows/phase_hampel/data.npz,yes,uses same split.csv
4,Temporal shift,raw + phase_hampel windows,raw_shift + phase_hampel_shift,yes,train only; val/test copied unchanged
5,Normalization stats,X_train per profile,normalization_stats.npz,sidecar,fit train only; no normalized data.npz
6,Dataset/DataLoader,data.npz + normalization_stats.npz,"(B, 192, 192)",runtime only,normalization + optional train noise


,scenario,profile,denoise,temporal_shift,noise_sigma,stored_data
0,1,raw,False,False,0.00,processed/windows/raw/data.npz
1,2,raw_shift,False,True,0.00,processed/windows/raw_shift/data.npz
2,3,raw_shift,False,True,0.01,processed/windows/raw_shift/data.npz
3,4,phase_hampel,True,False,0.00,processed/windows/phase_hampel/data.npz
4,5,phase_hampel_shift,True,True,0.00,processed/windows/phase_hampel_shift/data.npz
5,6,phase_hampel_shift,True,True,0.01,processed/windows/phase_hampel_shift/data.npz


## 1. Split + Window Raw

Build or refresh `processed/windows/raw/data.npz`. The fixed split is `processed/sample_split/split.csv`.

In [16]:
####### RUN RAW WINDOW
cmd = [
    sys.executable, str(SPLIT_WINDOW_SCRIPT),
    '--input-dir', str(RAW_SESSION_DIR),
    '--name', 'raw',
    '--processed-dir', str(PROCESSED_DIR),
    '--split-path', str(SPLIT_PATH),
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)


C:\Users\nguyen\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe c:\Users\nguyen\Desktop\DL_V2\Phần 1 tiền xử lí\split_and_window.py --input-dir c:\Users\nguyen\Desktop\DL_V2\Dataset_CSI_3D_v2\session1 --name raw --processed-dir c:\Users\nguyen\Desktop\DL_V2\Phần 1 tiền xử lí\processed --split-path c:\Users\nguyen\Desktop\DL_V2\Phần 1 tiền xử lí\processed\sample_split\split.csv


CompletedProcess(args=['C:\\Users\\nguyen\\AppData\\Local\\Microsoft\\WindowsApps\\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\\python.exe', 'c:\\Users\\nguyen\\Desktop\\DL_V2\\Phần 1 tiền xử lí\\split_and_window.py', '--input-dir', 'c:\\Users\\nguyen\\Desktop\\DL_V2\\Dataset_CSI_3D_v2\\session1', '--name', 'raw', '--processed-dir', 'c:\\Users\\nguyen\\Desktop\\DL_V2\\Phần 1 tiền xử lí\\processed', '--split-path', 'c:\\Users\\nguyen\\Desktop\\DL_V2\\Phần 1 tiền xử lí\\processed\\sample_split\\split.csv'], returncode=0)

## 2. Denoise

Main profile is `phase_hampel`. Keep `phase_hampel_dwt` as optional ablation.

In [17]:
####### RUN DENOISE
for profile in DENOISE_PROFILES:
    result = denoise.run_denoise(profile, overwrite=OVERWRITE_DENOISE)
    print(result)


RunResult(profile='phase_hampel', input_dir=WindowsPath('C:/Users/nguyen/Desktop/DL_V2/Dataset_CSI_3D_v2/session1'), output_dir=WindowsPath('C:/Users/nguyen/Desktop/DL_V2/Phần 1 tiền xử lí/processed/denoised/phase_hampel'), report_dir=WindowsPath('C:/Users/nguyen/Desktop/DL_V2/Phần 1 tiền xử lí/reports/denoise/phase_hampel'), processed_samples=931, processed_frames=278846)


## 3. Window Denoised Profiles

Use the same `split.csv`; do not rebuild split here.

In [18]:
####### RUN WINDOW FOR DENOISED DATA
for profile in DENOISE_PROFILES:
    input_dir = PROCESSED_DIR / 'denoised' / profile
    cmd = [
        sys.executable, str(SPLIT_WINDOW_SCRIPT),
        '--input-dir', str(input_dir),
        '--name', profile,
        '--processed-dir', str(PROCESSED_DIR),
        '--split-path', str(SPLIT_PATH),
    ]
    print(' '.join(cmd))
    subprocess.run(cmd, check=True)


C:\Users\nguyen\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe c:\Users\nguyen\Desktop\DL_V2\Phần 1 tiền xử lí\split_and_window.py --input-dir c:\Users\nguyen\Desktop\DL_V2\Phần 1 tiền xử lí\processed\denoised\phase_hampel --name phase_hampel --processed-dir c:\Users\nguyen\Desktop\DL_V2\Phần 1 tiền xử lí\processed --split-path c:\Users\nguyen\Desktop\DL_V2\Phần 1 tiền xử lí\processed\sample_split\split.csv


## 4. Temporal Shift Augmentation

Build train-only artifacts:
- `processed/windows/raw_shift/data.npz`
- `processed/windows/phase_hampel_shift/data.npz`

Current config: `stride=16`, `max_crops_per_sample=4`. Val/test are copied unchanged from the source profile.

In [19]:
####### RUN TEMPORAL SHIFT AUGMENTATION
results = augment.run_default_profiles(
    processed_dir=PROCESSED_DIR,
    split_path=SPLIT_PATH,
    stride=AUGMENT_STRIDE,
    max_crops_per_sample=AUGMENT_MAX_CROPS,
    overwrite=OVERWRITE_AUGMENT,
)
for result in results:
    print(result)


AugmentResult(source_name='raw', output_name='raw_shift', input_dir=WindowsPath('C:/Users/nguyen/Desktop/DL_V2/Dataset_CSI_3D_v2/session1'), output_dir=WindowsPath('C:/Users/nguyen/Desktop/DL_V2/Phần 1 tiền xử lí/processed/windows/raw_shift'), train_windows=2608, val_windows=140, test_windows=139, skipped_train_samples=0)
AugmentResult(source_name='phase_hampel', output_name='phase_hampel_shift', input_dir=WindowsPath('C:/Users/nguyen/Desktop/DL_V2/Phần 1 tiền xử lí/processed/denoised/phase_hampel'), output_dir=WindowsPath('C:/Users/nguyen/Desktop/DL_V2/Phần 1 tiền xử lí/processed/windows/phase_hampel_shift'), train_windows=2608, val_windows=140, test_windows=139, skipped_train_samples=0)


## 5. Smoke Check data.npz

Check shape, row counts, NaN/inf for the four main artifacts.

In [20]:
def profile_summary(name: str) -> dict[str, object]:
    window_dir = PROCESSED_DIR / 'windows' / name
    data_path = window_dir / 'data.npz'
    windows_path = window_dir / 'windows.csv'
    stats_path = window_dir / 'normalization_stats.npz'
    if not data_path.exists():
        return {'profile': name, 'status': 'missing'}
    with np.load(data_path) as data:
        shapes = {split: data[f'X_{split}'].shape for split in ['train', 'val', 'test']}
        has_nan = any(np.isnan(data[f'X_{split}']).any() for split in ['train', 'val', 'test'])
        has_inf = any(np.isinf(data[f'X_{split}']).any() for split in ['train', 'val', 'test'])
    windows = pd.read_csv(windows_path)
    counts = windows['split'].value_counts().to_dict()
    return {
        'profile': name,
        'status': 'ready',
        'train': int(counts.get('train', 0)),
        'val': int(counts.get('val', 0)),
        'test': int(counts.get('test', 0)),
        'X_train_shape': shapes['train'],
        'X_val_shape': shapes['val'],
        'X_test_shape': shapes['test'],
        'has_norm_stats': stats_path.exists(),
        'has_nan': bool(has_nan),
        'has_inf': bool(has_inf),
    }

##### ARTIFACT SUMMARY TABLE
artifact_df = pd.DataFrame([profile_summary(profile) for profile in WINDOW_PROFILES])
display(artifact_df)


,profile,status,train,val,test,X_train_shape,X_val_shape,X_test_shape,has_norm_stats,has_nan,has_inf
0,raw,ready,652,140,139,"(652, 3, 64, 192)","(140, 3, 64, 192)","(139, 3, 64, 192)",False,False,False
1,raw_shift,ready,2608,140,139,"(2608, 3, 64, 192)","(140, 3, 64, 192)","(139, 3, 64, 192)",False,False,False
2,phase_hampel,ready,652,140,139,"(652, 3, 64, 192)","(140, 3, 64, 192)","(139, 3, 64, 192)",False,False,False
3,phase_hampel_shift,ready,2608,140,139,"(2608, 3, 64, 192)","(140, 3, 64, 192)","(139, 3, 64, 192)",False,False,False


## 6. Confirm Val/Test Are Not Augmented

Shifted profiles must only change train.

In [21]:
def assert_val_test_unchanged(base: str, shifted: str):
    keys = ['X_val', 'X_test', 'y_pose_val', 'y_pose_test', 'y_cell_val', 'y_cell_test', 'y_presence_val', 'y_presence_test', 'y_center_val', 'y_center_test']
    with np.load(PROCESSED_DIR / 'windows' / base / 'data.npz') as base_data, np.load(PROCESSED_DIR / 'windows' / shifted / 'data.npz') as shifted_data:
        for key in keys:
            if not np.array_equal(base_data[key], shifted_data[key]):
                raise AssertionError(f'{shifted} differs from {base} at {key}')
    print(f'OK: {shifted} val/test unchanged from {base}')

assert_val_test_unchanged('raw', 'raw_shift')
assert_val_test_unchanged('phase_hampel', 'phase_hampel_shift')


OK: raw_shift val/test unchanged from raw
OK: phase_hampel_shift val/test unchanged from phase_hampel


## 7. Gaussian Noise Policy

Gaussian Noise is train-time only, applied after normalization inside the Dataset, and does not create another `data.npz`.

In [22]:
####### GAUSSIAN NOISE POLICY
print('Noise sigma configured for train Dataset only:', NOISE_SIGMA)
print('Actual noise behavior is checked in the Dataset/DataLoader smoke cell after normalization.')


Noise sigma configured for train Dataset only: 0.01
Actual noise behavior is checked in the Dataset/DataLoader smoke cell after normalization.


## 8. Normalization Sidecars

Fit z-score stats from `X_train` only for each profile. This creates `normalization_stats.npz` beside each existing `data.npz` and never rewrites `data.npz`.

In [23]:
##### FIT NORMALIZATION STATS
stats_paths = normalize.fit_profiles(PROCESSED_DIR / 'windows', WINDOW_PROFILES, eps=NORMALIZE_EPS)
for profile, path in stats_paths.items():
    stats = normalize.load_stats(path.parent)
    print(profile, path.name, 'mean', stats['mean'].shape, 'std', stats['std'].shape)


raw normalization_stats.npz mean (1, 3, 64, 1) std (1, 3, 64, 1)
raw_shift normalization_stats.npz mean (1, 3, 64, 1) std (1, 3, 64, 1)
phase_hampel normalization_stats.npz mean (1, 3, 64, 1) std (1, 3, 64, 1)
phase_hampel_shift normalization_stats.npz mean (1, 3, 64, 1) std (1, 3, 64, 1)


## 9. Inspect Normalized Stats

Quick inspection that train normalization is centered/scaled per profile. Val/test use the train stats at runtime.

In [24]:
##### NORMALIZATION SUMMARY TABLE
normalization_rows = []
for profile in WINDOW_PROFILES:
    profile_dir = PROCESSED_DIR / 'windows' / profile
    stats = normalize.load_stats(profile_dir)
    train_stats = normalize.normalized_split_stats(profile_dir, split='train')
    normalization_rows.append({
        'profile': profile,
        'mean_shape': stats['mean'].shape,
        'std_shape': stats['std'].shape,
        'train_mean': train_stats['mean'],
        'global_train_std': train_stats['std'],
        'active_feature_std': train_stats['active_feature_std'],
        'constant_features': int(train_stats['constant_features']),
        'min': train_stats['min'],
        'max': train_stats['max'],
    })
normalization_df = pd.DataFrame(normalization_rows)
display(normalization_df)


,profile,mean_shape,std_shape,train_mean,global_train_std,active_feature_std,constant_features,min,max
0,raw,"(1, 3, 64, 1)","(1, 3, 64, 1)",5.200678e-09,0.935414,1.0,24,-4.072886,10.372913
1,raw_shift,"(1, 3, 64, 1)","(1, 3, 64, 1)",1.040136e-08,0.935414,1.0,24,-4.070456,10.368912
2,phase_hampel,"(1, 3, 64, 1)","(1, 3, 64, 1)",0.000000e+00,0.935414,1.0,24,-4.216776,9.482571
3,phase_hampel_shift,"(1, 3, 64, 1)","(1, 3, 64, 1)",1.040136e-08,0.935414,1.0,24,-4.214540,9.475681


## 10. Dataset/DataLoader Smoke Check

Dataset reads existing profile `data.npz`, applies normalization at runtime, applies Gaussian noise only for train when `NOISE_SIGMA > 0`, and returns GRU layout `(192, 192)`.

In [25]:
##### DATALOADER PREVIEW TABLE
loader = csi_dataset.make_dataloader(
    windows_dir=PROCESSED_DIR / 'windows',
    profile=DATASET_PROFILE,
    split='train',
    target=TARGET,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    noise_sigma=NOISE_SIGMA,
)
xb, yb = next(iter(loader))
loader_df = pd.DataFrame([{
    'profile': DATASET_PROFILE,
    'split': 'train',
    'target': TARGET,
    'noise_sigma': NOISE_SIGMA,
    'batch_size': BATCH_SIZE,
    'x_batch_shape': tuple(xb.shape),
    'x_dtype': str(xb.dtype),
    'y_batch_shape': tuple(yb.shape),
    'y_dtype': str(yb.dtype),
    'model_ready_shape': '(B, 192, 192)',
}])
display(loader_df)

display(pd.DataFrame([
    {'scenario': index + 1, 'profile': profile, 'noise_sigma': noise, 'loader_x_shape': '(B, 192, 192)'}
    for index, (profile, noise) in enumerate(TRAINING_SCENARIOS)
]))

if TARGET != 'center':
    distribution = csi_dataset.class_distribution(PROCESSED_DIR / 'windows' / DATASET_PROFILE, 'train', TARGET)
    weights = csi_dataset.class_weights(PROCESSED_DIR / 'windows' / DATASET_PROFILE, 'train', TARGET).tolist()
    imbalance_df = pd.DataFrame([
        {'target': TARGET, 'class_id': class_id, 'train_count': count, 'class_weight': weights[class_id]}
        for class_id, count in distribution.items()
    ])
    display(imbalance_df)
else:
    print('class distribution/weights skipped for regression target: center')


,profile,split,target,noise_sigma,batch_size,x_batch_shape,x_dtype,y_batch_shape,y_dtype,model_ready_shape
0,raw_shift,train,pose,0.01,8,"(8, 192, 192)",torch.float32,"(8,)",torch.int64,"(B, 192, 192)"


,scenario,profile,noise_sigma,loader_x_shape
0,1,raw,0.00,"(B, 192, 192)"
1,2,raw_shift,0.00,"(B, 192, 192)"
2,3,raw_shift,0.01,"(B, 192, 192)"
3,4,phase_hampel,0.00,"(B, 192, 192)"
4,5,phase_hampel_shift,0.00,"(B, 192, 192)"
5,6,phase_hampel_shift,0.01,"(B, 192, 192)"


,target,class_id,train_count,class_weight
0,pose,0,252,1.478458
1,pose,1,152,2.451128
2,pose,2,320,1.164286
3,pose,3,588,0.633625
4,pose,4,876,0.425310
5,pose,5,152,2.451128
6,pose,6,268,1.390192
